# D2 — Retrieval Stack & Knowledge Graph

This notebook demonstrates the complete D2 pipeline:

1. **PDF Ingestion** — PyMuPDF extraction, text chunking (500 char / 100 overlap), embedding (BGE), storage to MongoDB + Qdrant
2. **Hybrid Search** — BM25 (sparse) + Dense (Qdrant) + Reciprocal Rank Fusion
3. **Knowledge Graph** — Neo4j with Paper, Author, Topic nodes and WROTE, HAS_TOPIC, CITES relationships
4. **Evaluation** — Recall@K, MRR, nDCG@K metrics

**Prerequisites:** MongoDB, Qdrant, and Neo4j must be running via `docker compose up -d mongodb qdrant neo4j`.

## 0. Setup & Imports

In [1]:
import sys, os

# Ensure the project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

print(f"Working directory: {os.getcwd()}")

Working directory: E:\PythonProject\PDF-Paper-AI-Agent


---
## 1. PDF Ingestion Pipeline

The ingestion pipeline extracts text from PDFs using PyMuPDF, chunks it into 500-character segments with 100-character overlap, generates 384-dim embeddings using `BAAI/bge-small-en-v1.5`, and stores:
- **MongoDB**: chunk metadata (chunk_id, paper_id, title, authors, text, page range)
- **Qdrant**: dense vector index for cosine similarity search

In [2]:
from ingest import (
    extract_text_with_pages,
    chunk_text_with_pages,
    embed_texts,
    ingest_directory,
    get_embedding_model,
    EMBEDDING_MODEL,
    EMBEDDING_DIM,
    MONGO_URI,
    MONGO_DB,
    QDRANT_HOST,
    QDRANT_PORT,
)

print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Embedding dim   : {EMBEDDING_DIM}")
print(f"MongoDB URI     : {MONGO_URI}")
print(f"Qdrant          : {QDRANT_HOST}:{QDRANT_PORT}")

E:\PythonProject\PDF-Paper-AI-Agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embedding model : BAAI/bge-small-en-v1.5
Embedding dim   : 384
MongoDB URI     : mongodb://localhost:27017
Qdrant          : localhost:6333


### 1.1 Ingest PDFs from the `papers/` directory

If `papers/` is empty, run `python seed_data.py` first to download sample arXiv PDFs.

In [3]:
import glob

pdfs = glob.glob("papers/*.pdf")
print(f"Found {len(pdfs)} PDFs in papers/:")
for p in pdfs:
    print(f"  - {os.path.basename(p)}")

if pdfs:
    chunks = ingest_directory("papers", batch_size=32)
    print(f"\nIngested {len(chunks)} total chunks.")
else:
    print("\nNo PDFs found. Run `python seed_data.py` first.")

2026-06-21 19:13:23,386 | INFO | Found 5 PDFs in 'papers'


Found 5 PDFs in papers/:
  - 1706.03762.pdf
  - 1810.04805.pdf
  - 2005.11401.pdf
  - 2302.13971.pdf
  - 2304.08485.pdf


2026-06-21 19:13:23,754 | INFO | HTTP Request: GET http://localhost:6333/collections "HTTP/1.1 200 OK"


2026-06-21 19:13:23,757 | INFO | Skipping '1706.03762.pdf' -- already ingested.


2026-06-21 19:13:23,758 | INFO | Skipping '1810.04805.pdf' -- already ingested.


2026-06-21 19:13:23,758 | INFO | Skipping '2005.11401.pdf' -- already ingested.


2026-06-21 19:13:23,759 | INFO | Skipping '2302.13971.pdf' -- already ingested.


2026-06-21 19:13:23,759 | INFO | Skipping '2304.08485.pdf' -- already ingested.


2026-06-21 19:13:23,759 | INFO | Ingestion complete -- 0 total chunks from 5 PDFs



Ingested 0 total chunks.


### 1.2 Inspect chunking

Let's look at what a chunked document looks like in MongoDB.

In [4]:
from pymongo import MongoClient

client = MongoClient(MONGO_URI)
col = client[MONGO_DB]["chunks"]

total = col.count_documents({})
print(f"Total chunks in MongoDB: {total}")

# Show first 3 chunks
print("\n--- Sample chunks ---")
for doc in col.find({}, {"_id": 0}).limit(3):
    print(f"\nchunk_id   : {doc['chunk_id']}")
    print(f"paper_id   : {doc['paper_id']}")
    print(f"title      : {doc.get('title', 'N/A')}")
    print(f"authors    : {doc.get('authors', [])}")
    print(f"pages      : {doc.get('page_start', '?')}-{doc.get('page_end', '?')}")
    print(f"text       : {doc['text'][:200]}...")

Total chunks in MongoDB: 872

--- Sample chunks ---

chunk_id   : bb468737-9d01-56b3-afc4-a25bc6d276aa
paper_id   : 1706.03762
title      : Provided proper attribution is provided, Google hereby grants permission to
authors    : ['reproduce the tables', 'figures in this paper solely for use in journalistic or', 'Attention Is All You Need', 'Ashish Vaswani∗', 'Google Brain']
pages      : 1-1
text       : Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...

chunk_id   : 5af12318-973b-5386-ad96-b8f322e80fef
paper_id   : 1706.03762
title      : Provided proper attribution is provided, Google hereby grants permission to
authors    : ['reproduce the tables', 'figures in this paper solely for use in journalistic or', 'Attention Is All You Need', 'Ashish Vaswani∗', 'Google Brain']
pages      : 1-1
text       : oogle Research
llion@google.com
Ai

### 1.3 Embedding demo

The embedding model is `BAAI/bge-small-en-v1.5` producing 384-dim normalized vectors.

In [5]:
import numpy as np

model = get_embedding_model()
sample_texts = ["attention mechanism", "knowledge graph", "language model"]
embeddings = model.encode(sample_texts, normalize_embeddings=True)

print(f"Shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")
print(f"L2 norm of first vector: {np.linalg.norm(embeddings[0]):.4f}")
print(f"\nFirst 10 dims of 'attention mechanism': {embeddings[0][:10]}")

2026-06-21 19:13:23,786 | INFO | Loading embedding model 'BAAI/bge-small-en-v1.5' ...


2026-06-21 19:13:23,789 | INFO | No device provided, using cpu


2026-06-21 19:13:24,162 | INFO | HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"


2026-06-21 19:13:24,777 | INFO | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


2026-06-21 19:13:24,902 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"


...

2026-06-21 19:13:30,831 | INFO | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5 "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.37it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.36it/s]

Shape: (3, 384)
Dtype: float32
L2 norm of first vector: 1.0000

First 10 dims of 'attention mechanism': [-0.0286588  -0.04473739  0.03841925 -0.02203975 -0.01045368 -0.00067347
  0.11368118  0.00794682  0.05964772 -0.00130284]


---
## 2. Hybrid Search (BM25 + Dense + RRF)

The `HybridSearcher` combines:
- **BM25** (sparse lexical matching via `rank-bm25`)
- **Dense** (cosine similarity via Qdrant)
- **RRF** (Reciprocal Rank Fusion with k=60)

Default parameters are loaded from `configs/run_card.yaml` (AutoML winning config: k=11, alpha=0.1154).

In [6]:
from hybrid_search import HybridSearcher, CONFIG_K, CONFIG_ALPHA

print(f"Config loaded from run_card.yaml:")
print(f"  k (top_k default) : {CONFIG_K}")
print(f"  alpha (BM25 wt)   : {CONFIG_ALPHA}")

searcher = HybridSearcher()

2026-06-21 19:13:31,315 | INFO | Loaded winning config from E:\PythonProject\PDF-Paper-AI-Agent\configs\run_card.yaml: k=11, alpha=0.1154


Config loaded from run_card.yaml:
  k (top_k default) : 11
  alpha (BM25 wt)   : 0.1154


### 2.1 BM25 Search (Sparse)

In [7]:
query = "attention mechanism in transformers"

bm25_results = searcher.bm25_search(query, top_k=5)
print(f"BM25 results for: '{query}'\n")
for i, r in enumerate(bm25_results, 1):
    print(f"  [{i}] score={r.score:.4f} | {r.title}")
    print(f"      pp. {r.page_start}-{r.page_end} | {r.text[:100]}...\n")

2026-06-21 19:13:31,671 | INFO | Building BM25 index from MongoDB ...


2026-06-21 19:13:31,759 | INFO | BM25 index built with 872 documents.


BM25 results for: 'attention mechanism in transformers'

  [1] score=13.0347 | Provided proper attribution is provided, Google hereby grants permission to
      pp. 2-2 | d in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relatin...

  [2] score=10.6555 | Provided proper attribution is provided, Google hereby grants permission to
      pp. 2-2 |  the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it...

  [3] score=9.2624 | Provided proper attribution is provided, Google hereby grants permission to
      pp. 13-13 | jority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
pro...

  [4] score=9.1055 | BERT: Pre-training of Deep Bidirectional Transformers for
      pp. 5-5 | he text passages
and ignore lists, tables, and headers. It is criti-
cal to use a document-level cor...

  [5] score=8.6830 | Provided proper attribution is provided, Google hereby grants pe

### 2.2 Dense Search (Qdrant)

In [8]:
dense_results = searcher.dense_search(query, top_k=5)
print(f"Dense results for: '{query}'\n")
for i, r in enumerate(dense_results, 1):
    print(f"  [{i}] score={r.score:.4f} | {r.title}")
    print(f"      pp. {r.page_start}-{r.page_end} | {r.text[:100]}...\n")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 64.40it/s]


2026-06-21 19:13:31,815 | INFO | HTTP Request: POST http://localhost:6333/collections/paper_chunks/points/query "HTTP/1.1 200 OK"


Dense results for: 'attention mechanism in transformers'

  [1] score=0.8086 | Provided proper attribution is provided, Google hereby grants permission to
      pp. 2-2 |  of sequential computation, however, remains.
Attention mechanisms have become an integral part of c...

  [2] score=0.7842 | Provided proper attribution is provided, Google hereby grants permission to
      pp. 1-1 | decoder through an attention
mechanism. We propose a new simple network architecture, the Transforme...

  [3] score=0.7778 | Provided proper attribution is provided, Google hereby grants permission to
      pp. 2-2 |  the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it...

  [4] score=0.7712 | Provided proper attribution is provided, Google hereby grants permission to
      pp. 2-2 | 7, 28, 22].
End-to-end memory networks are based on a recurrent attention mechanism instead of seque...

  [5] score=0.7615 | Provided proper attribution is provided, Google h

### 2.3 Hybrid Search (RRF Fusion)

Reciprocal Rank Fusion merges both ranked lists:

$$\text{RRF}(d) = \sum_{i} \frac{1}{k + \text{rank}_i(d)}, \quad k=60$$

In [9]:
hybrid_results = searcher.search(query, top_k=5)
print(f"Hybrid (RRF) results for: '{query}'\n")
for i, r in enumerate(hybrid_results, 1):
    print(f"  [{i}] rrf_score={r.score:.6f} | {r.title}")
    print(f"      Citation: {r.citation()}")
    print(f"      Text: {r.text[:120]}...\n")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 71.42it/s]


2026-06-21 19:13:31,862 | INFO | HTTP Request: POST http://localhost:6333/collections/paper_chunks/points/query "HTTP/1.1 200 OK"


Hybrid (RRF) results for: 'attention mechanism in transformers'

  [1] rrf_score=0.032002 | Provided proper attribution is provided, Google hereby grants permission to
      Citation: [reproduce the tables, figures in this paper solely for use in journalistic or, Attention Is All You Need et al.] "Provided proper attribution is provided, Google hereby grants permission to" (pp. 2-2)
      Text:  the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to l...

  [2] rrf_score=0.031010 | Provided proper attribution is provided, Google hereby grants permission to
      Citation: [reproduce the tables, figures in this paper solely for use in journalistic or, Attention Is All You Need et al.] "Provided proper attribution is provided, Google hereby grants permission to" (pp. 2-2)
      Text: 7, 28, 22].
End-to-end memory networks are based on a recurrent attention mechanism instead of sequence-
aligned recurre...

  [3] rrf_score=0.0

### 2.4 Filtered Search (paper_ids)

The search methods accept an optional `paper_ids` parameter for graph-guided retrieval (used by D3).

In [10]:
# Get one paper_id from the corpus to demonstrate filtering
sample_doc = col.find_one({}, {"paper_id": 1, "title": 1, "_id": 0})
if sample_doc:
    target_id = sample_doc["paper_id"]
    print(f"Filtering to paper: {sample_doc.get('title', target_id)}")
    print(f"paper_id: {target_id}\n")

    filtered = searcher.search(query, top_k=5, paper_ids=[target_id])
    print(f"Filtered results ({len(filtered)} chunks):")
    for i, r in enumerate(filtered, 1):
        print(f"  [{i}] {r.score:.6f} | {r.title} | pp. {r.page_start}-{r.page_end}")
else:
    print("No documents in MongoDB. Run seed_data.py first.")

Filtering to paper: Provided proper attribution is provided, Google hereby grants permission to
paper_id: 1706.03762



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 78.06it/s]


2026-06-21 19:13:31,962 | INFO | HTTP Request: POST http://localhost:6333/collections/paper_chunks/points/query "HTTP/1.1 200 OK"


Filtered results (5 chunks):
  [1] 0.032002 | Provided proper attribution is provided, Google hereby grants permission to | pp. 2-2
  [2] 0.031250 | Provided proper attribution is provided, Google hereby grants permission to | pp. 2-2
  [3] 0.031099 | Provided proper attribution is provided, Google hereby grants permission to | pp. 2-2
  [4] 0.030090 | Provided proper attribution is provided, Google hereby grants permission to | pp. 2-2
  [5] 0.028950 | Provided proper attribution is provided, Google hereby grants permission to | pp. 1-1


---
## 3. Neo4j Knowledge Graph

The knowledge graph stores three node types and three relationship types:

```
Author --(WROTE)--> Paper --(HAS_TOPIC)--> Topic
                    Paper --(CITES)------> Paper
```

In [11]:
from graph_build import KnowledgeGraph, populate_from_mongodb, extract_topics

graph = KnowledgeGraph()

2026-06-21 19:13:32,002 | INFO | HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"


2026-06-21 19:13:32,131 | INFO | Connected to Neo4j at bolt://localhost:7687


### 3.1 Populate graph from MongoDB

In [12]:
populate_from_mongodb(graph)

stats = graph.stats()
print("\nGraph statistics:")
for key, val in stats.items():
    print(f"  {key}: {val}")

2026-06-21 19:13:32,414 | INFO | Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description="note: successful completion - index or constraint already exists. The command 'CREATE CONSTRAINT paper_id IF NOT EXISTS FOR (e:Paper) REQUIRE (e.paper_id) IS UNIQUE' has no effect. The index or constraint specified by 'CONSTRAINT paper_id FOR (e:Paper) REQUIRE (e.paper_id) IS UNIQUE' already exists.", position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'SCHEMA', '_severity': 'INFORMATION', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CREATE CONSTRAINT paper_id IF NOT EXISTS FOR (p:Paper) REQUIRE p.paper_id IS UNIQUE'


2026-06-21 19:13:32,436 | INFO | Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description="note: successful completion - index or constraint already exists. The command 'CREATE CONSTRAINT author_name IF NOT EXISTS FOR (e:Author) REQUIRE (e.name) IS UNIQUE' has no effect. The index or constraint specified by 'CONSTRAINT author_name FOR (e:Author) REQUIRE (e.name) IS UNIQUE' already exists.", position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'SCHEMA', '_severity': 'INFORMATION', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CREATE CONSTRAINT author_name IF NOT EXISTS FOR (a:Author) REQUIRE a.name IS UNIQUE'


2026-06-21 19:13:32,452 | INFO | Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description="note: successful completion - index or constraint already exists. The command 'CREATE CONSTRAINT topic_name IF NOT EXISTS FOR (e:Topic) REQUIRE (e.name) IS UNIQUE' has no effect. The index or constraint specified by 'CONSTRAINT topic_name FOR (e:Topic) REQUIRE (e.name) IS UNIQUE' already exists.", position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'SCHEMA', '_severity': 'INFORMATION', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CREATE CONSTRAINT topic_name IF NOT EXISTS FOR (t:Topic) REQUIRE t.name IS UNIQUE'


2026-06-21 19:13:32,480 | INFO | Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description="note: successful completion - index or constraint already exists. The command 'CREATE RANGE INDEX paper_title IF NOT EXISTS FOR (e:Paper) ON (e.title)' has no effect. The index or constraint specified by 'RANGE INDEX paper_title FOR (e:Paper) ON (e.title)' already exists.", position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'SCHEMA', '_severity': 'INFORMATION', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CREATE INDEX paper_title IF NOT EXISTS FOR (p:Paper) ON (p.title)'


2026-06-21 19:13:32,481 | INFO | Constraints and indexes created.


2026-06-21 19:13:32,493 | INFO | Found 5 unique papers in MongoDB.


2026-06-21 19:13:34,586 | WARNING | Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `CITES` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=13, offset=12>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 12, 'line': 1, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH ()-[r:CITES]->() RETURN count(r) AS n'


2026-06-21 19:13:34,586 | INFO | Graph populated -- {'papers': 5, 'authors': 46, 'topics': 3, 'wrote_edges': 46, 'topic_edges': 6, 'cites_edges': 0}


2026-06-21 19:13:34,607 | WARNING | Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `CITES` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=13, offset=12>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 12, 'line': 1, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH ()-[r:CITES]->() RETURN count(r) AS n'



Graph statistics:
  papers: 5
  authors: 46
  topics: 3
  wrote_edges: 46
  topic_edges: 6
  cites_edges: 0


### 3.2 Topic extraction demo

Topics are extracted via keyword matching against 10 categories.

In [13]:
sample_text = "We propose a new transformer architecture for natural language processing using self-attention."
topics = extract_topics(sample_text)
print(f"Text: {sample_text}")
print(f"Extracted topics: {topics}")

Text: We propose a new transformer architecture for natural language processing using self-attention.
Extracted topics: ['Natural Language Processing']


### 3.3 Five Cypher queries

In [14]:
# Query 1: Topic distribution
print("Topic Distribution:")
for row in graph.count_papers_per_topic():
    print(f"  {row['topic']}: {row['paper_count']} papers")

Topic Distribution:
  Natural Language Processing: 4 papers
  General: 1 papers
  Information Retrieval: 1 papers


In [15]:
# Query 2: Papers by topic
topic = "Natural Language Processing"
papers = graph.find_papers_by_topic(topic)
print(f"\nPapers about '{topic}':")
for p in papers:
    print(f"  - {p['title']}")


Papers about 'Natural Language Processing':
  - BERT: Pre-training of Deep Bidirectional Transformers for
  - LLaMA: Open and Efﬁcient Foundation Language Models
  - Retrieval-Augmented Generation for
  - Visual Instruction Tuning


In [16]:
# Query 3: Find related papers (shared topics)
if papers:
    pid = papers[0]["paper_id"]
    related = graph.find_related_papers(pid)
    print(f"\nPapers related to '{papers[0]['title']}':")
    for r in related:
        print(f"  - {r['title']} (shared topics: {r['shared_topics']})")
else:
    print("No papers found for this topic.")


Papers related to 'BERT: Pre-training of Deep Bidirectional Transformers for':
  - Visual Instruction Tuning (shared topics: ['Natural Language Processing'])
  - Retrieval-Augmented Generation for (shared topics: ['Natural Language Processing'])
  - LLaMA: Open and Efﬁcient Foundation Language Models (shared topics: ['Natural Language Processing'])


---
## 4. Evaluation Metrics

D2 includes standard IR metrics: **Recall@K**, **MRR**, and **nDCG@K**.

In [17]:
from hybrid_search import recall_at_k, mrr, ndcg_at_k, run_quick_eval

# Synthetic example
retrieved = ["doc_a", "doc_b", "doc_c", "doc_d", "doc_e"]
relevant = {"doc_c", "doc_e"}

print(f"Retrieved : {retrieved}")
print(f"Relevant  : {relevant}")
print(f"Recall@5  : {recall_at_k(retrieved, relevant, k=5):.3f}")
print(f"MRR       : {mrr(retrieved, relevant):.3f}")
print(f"nDCG@5    : {ndcg_at_k(retrieved, relevant, k=5):.3f}")

Retrieved : ['doc_a', 'doc_b', 'doc_c', 'doc_d', 'doc_e']
Relevant  : {'doc_e', 'doc_c'}
Recall@5  : 1.000
MRR       : 0.333
nDCG@5    : 0.544


### 4.1 Live evaluation against seeded data

In [18]:
# Build eval queries from papers in MongoDB
pipeline = [
    {"$group": {"_id": "$paper_id", "title": {"$first": "$title"}}},
    {"$limit": 4},
]
paper_docs = list(col.aggregate(pipeline))

eval_queries = []
for doc in paper_docs:
    title = doc.get("title", "")
    if title:
        eval_queries.append({
            "query": title,
            "relevant_ids": [doc["_id"]],
        })

if eval_queries:
    run_quick_eval(searcher, eval_queries)
else:
    print("No papers in MongoDB for evaluation.")


 Query                            Recall@5        MRR     nDCG@5
------------------------------------------------------------------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 18.61it/s]


2026-06-21 19:13:35,160 | INFO | HTTP Request: POST http://localhost:6333/collections/paper_chunks/points/query "HTTP/1.1 200 OK"


 Provided proper attribution...      1.000      1.000      1.000


...

 Retrieval-Augmented Generat...      1.000      1.000      1.000


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 58.82it/s]


2026-06-21 19:13:35,249 | INFO | HTTP Request: POST http://localhost:6333/collections/paper_chunks/points/query "HTTP/1.1 200 OK"


 LLaMA: Open and Efﬁcient Fo...      1.000      1.000      1.000
------------------------------------------------------------------------
 MEAN                                1.000      1.000      0.976



---
## 5. FastAPI Integration

All D2 components are exposed via the FastAPI app in `app.py`:

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/ingest` | POST | Ingest PDFs |
| `/search` | POST | Hybrid BM25+Dense+RRF search |
| `/graph/query` | POST | Raw Cypher queries |
| `/stats` | GET | System statistics |
| `/health` | GET | Service health check |

Start the server with: `uvicorn app:app --host 0.0.0.0 --port 8000 --reload`

In [19]:
import requests

BASE = "http://localhost:8000"

try:
    health = requests.get(f"{BASE}/health", timeout=3).json()
    print("Health check:")
    for k, v in health.items():
        print(f"  {k}: {v}")

    resp = requests.post(f"{BASE}/search", json={
        "query": "attention mechanism",
        "top_k": 3,
    }, timeout=30).json()
    print(f"\n/search returned {resp['num_results']} results in {resp['elapsed_ms']:.1f}ms")
    for r in resp["results"]:
        print(f"  - {r['citation']}")
except requests.ConnectionError:
    print("API server not running. Start it with: uvicorn app:app --port 8000")

Health check:
  status: healthy
  mongo: ok
  qdrant: ok
  neo4j: ok



/search returned 3 results in 7640.5ms
  - [reproduce the tables, figures in this paper solely for use in journalistic or, Attention Is All You Need et al.] "Provided proper attribution is provided, Google hereby grants permission to" (pp. 2-2)
  - [reproduce the tables, figures in this paper solely for use in journalistic or, Attention Is All You Need et al.] "Provided proper attribution is provided, Google hereby grants permission to" (pp. 2-2)
  - [reproduce the tables, figures in this paper solely for use in journalistic or, Attention Is All You Need et al.] "Provided proper attribution is provided, Google hereby grants permission to" (pp. 4-4)


---
## Cleanup

In [20]:
graph.close()
print("Neo4j connection closed.")
print("\nD2 notebook complete.")

Neo4j connection closed.

D2 notebook complete.
